# Caderno 02: Cruzamento Espacial Estático e Gabaritos

## Referencial Teórico: Anthony Downs (1957)
No livro clássico *"An Economic Theory of Democracy"*, Downs introduziu a **Teoria Espacial do Voto**. Ele sugere que eleitores e partidos podem ser alocados em um espaço ideológico (tradicionalmente um eixo linear Esquerda-Direita, mas expansível para N-dimensões).

O eleitor racional votará no candidato que estiver **geometricamente mais próximo** de si neste espaço. Se o eleitor está no ponto X, e os candidatos estão nos pontos A e B, o cálculo da distância determinará o voto.

Neste caderno, faremos exatamente isso usando vetores matemáticos e Distância Euclidiana em todo o nosso DataSet bruto.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
sns.set_theme(style='whitegrid')

## 1. O Mapa de Ideologias (Partidos vs Políticos)
O sistema lida com o fenômeno em que as ações de um político podem desviar do estatuto original de seu partido. Por isso, vetorizamos ambas as entidades em 7 dimensões.


In [ ]:
partidos_data = {
    'Partido': ['PT', 'PL', 'NOVO', 'PDT'],
    'ideologia': ['Social-Democracia', 'Centrão/Fisiológico', 'Liberalismo Clássico', 'Trabalhismo'],
    'justificativa': ['Foco no estado de bem-estar social', 'Foco na governabilidade', 'Foco em estado mínimo', 'Defesa da indústria nacional'],
    'bloco_d_estatais': [2, 4, 5, 2],
    'bloco_d_tabelamento': [3, 2, 1, 3],
    'bloco_e_punitivismo': [2, 4, 3, 3],
    'bloco_e_educacao': [2, 4, 3, 2],
    'bloco_g_corrupcao': [2, 4, 5, 2],
    'bloco_g_pesquisa': [4, 2, 1, 5],
    'bloco_g_politica_externa': [4, 2, 1, 4]
}
df_partidos = pd.DataFrame(partidos_data).set_index('Partido')

politicos_data = {
    'Político': ['Lula', 'Flavio Bolsonaro', 'Romeu Zema', 'Ciro Gomes'],
    'ideologia': ['Pragmatismo Progressista', 'Direita Radical', 'Neoliberalismo', 'Nacional-Desenvolvimentismo'],
    'justificativa': ['Conciliação de pautas', 'Pauta de costumes', 'Livre mercado', 'Estado forte na economia'],
    'bloco_d_estatais': [1, 5, 5, 2],
    'bloco_d_tabelamento': [4, 1, 1, 3],
    'bloco_e_punitivismo': [2, 5, 4, 3],
    'bloco_e_educacao': [2, 5, 4, 2],
    'bloco_g_corrupcao': [1, 5, 5, 2],
    'bloco_g_pesquisa': [5, 1, 2, 5],
    'bloco_g_politica_externa': [5, 1, 2, 4]
}
df_politicos = pd.DataFrame(politicos_data).set_index('Político')

cols_opiniao = ['bloco_d_estatais', 'bloco_d_tabelamento', 'bloco_e_punitivismo', 'bloco_e_educacao', 'bloco_g_corrupcao', 'bloco_g_pesquisa', 'bloco_g_politica_externa']
display(df_partidos[cols_opiniao])
display(df_politicos[cols_opiniao])

,bloco_d_estatais,bloco_d_tabelamento,bloco_e_punitivismo,bloco_e_educacao,bloco_g_corrupcao,bloco_g_pesquisa,bloco_g_politica_externa
Partido,,,,,,,
PT,2,3,2,2,2,4,4
PL,4,2,4,4,4,2,2
NOVO,5,1,3,3,5,1,1
PDT,2,3,3,2,2,5,4


,bloco_d_estatais,bloco_d_tabelamento,bloco_e_punitivismo,bloco_e_educacao,bloco_g_corrupcao,bloco_g_pesquisa,bloco_g_politica_externa
Político,,,,,,,
Lula,1,4,2,2,1,5,5
Flavio Bolsonaro,5,1,5,5,5,1,1
Romeu Zema,5,1,4,4,5,2,2
Ciro Gomes,2,3,3,2,2,5,4


## 2. A Matemática: Distância Euclidiana
A fórmula da distância euclidiana mede o comprimento do segmento de reta entre o eleitor e o candidato no hiper-espaço de 7 dimensões. Quanto **menor** a distância, maior a chance de voto.


In [ ]:
df_eleitores = pd.read_csv('../data/raw/mock_voters.csv')
vetores_eleitores = df_eleitores[cols_opiniao].head(5).values

dist_pol = euclidean_distances(vetores_eleitores, df_politicos[cols_opiniao].values)
dist_part = euclidean_distances(vetores_eleitores, df_partidos[cols_opiniao].values)

print('Distância Euclidiana (Amostra de 5 Eleitores x Políticos):')
print(pd.DataFrame(dist_pol, columns=df_politicos.index))

# O Argmin revela a menor distância (A recomendação de fato)
eleitores_recomendados = [df_politicos.index[i] for i in np.argmin(dist_pol, axis=1)]
print('\nRecomendação Puramente Matemática para os 5 primeiros:', eleitores_recomendados)

Distância Euclidiana (Amostra de 5 Eleitores x Políticos):
Político      Lula  Flavio Bolsonaro  Romeu Zema  Ciro Gomes
0         5.830952          7.280110    6.403124    4.123106
1         3.741657          7.141428    5.916080    2.645751
2         5.477226          5.916080    4.582576    3.605551
3         7.348469          4.582576    3.872983    6.082763
4         7.416198          3.162278    3.162278    5.656854

Recomendação Puramente Matemática para os 5 primeiros: ['Ciro Gomes', 'Ciro Gomes', 'Ciro Gomes', 'Romeu Zema', 'Flavio Bolsonaro']
